# Transformer Differential and REF Protection Calculation
**Musina 33/11 kV — 20 MVA Dyn11 Transformer**  
**Brownfield Protection Review**  
**Standards: IEC 60255‑8, NRS 034**

In [1]:
import numpy as np

S_mva  = 20.0       # MVA
V_hv   = 33.0       # kV
V_lv   = 11.0       # kV

I_hv_rated = S_mva * 1000.0 / (np.sqrt(3.0) * V_hv)
I_lv_rated = S_mva * 1000.0 / (np.sqrt(3.0) * V_lv)

print(f"HV rated current: {I_hv_rated:.1f} A")
print(f"LV rated current: {I_lv_rated:.1f} A")

HV rated current: 349.9 A
LV rated current: 1049.7 A


In [3]:
# Selected CT ratios to match secondary currents
CT_hv = 200        # 200/1
CT_lv = 1200       # 1200/1

I_hv_sec_rated = I_hv_rated / CT_hv
I_lv_sec_rated = I_lv_rated / CT_lv

print(f"HV CT {CT_hv}/1: secondary rated = {I_hv_sec_rated:.3f} A")
print(f"LV CT {CT_lv}/1: secondary rated = {I_lv_sec_rated:.3f} A")
print(f"Secondary current mismatch: {abs(I_hv_sec_rated - I_lv_sec_rated)*100/I_hv_sec_rated:.1f} %")
print("(Mismatch handled by relay software; Dyn11 phase shift correction required.)")

HV CT 200/1: secondary rated = 1.750 A
LV CT 1200/1: secondary rated = 0.875 A
Secondary current mismatch: 50.0 %
(Mismatch handled by relay software; Dyn11 phase shift correction required.)


In [4]:
# Differential settings according to SEL‑387 / typical numeric relay
Idiff_pickup = 0.20   # pu of rated LV secondary (minimum pickup)
slope_1 = 0.20        # 20% slope for low bias
slope_2 = 0.60        # 60% slope for high bias (CT saturation region)
I_bias_breakpoint = 2.0  # pu rated (point where slope changes)

# Idiff pickup in primary Amps
Idiff_prim = Idiff_pickup * I_lv_rated
print("=== Differential Protection Settings ===")
print(f"  Idiff> pickup: {Idiff_pickup} pu ({Idiff_prim:.0f} A primary at LV side)")
print(f"  Slope 1: {slope_1*100:.0f} %  (up to {I_bias_breakpoint} pu bias)")
print(f"  Slope 2: {slope_2*100:.0f} %  (above {I_bias_breakpoint} pu bias)")
print(f"  2nd harmonic blocking: 15% (inrush restraint)")

# Minimum detectable internal fault (LV side)
I_min_fault = Idiff_pickup * I_lv_rated
print(f"\nMinimum detectable internal fault (LV): {I_min_fault:.0f} A ({Idiff_pickup*100:.0f} % of rated)")

=== Differential Protection Settings ===
  Idiff> pickup: 0.2 pu (210 A primary at LV side)
  Slope 1: 20 %  (up to 2.0 pu bias)
  Slope 2: 60 %  (above 2.0 pu bias)
  2nd harmonic blocking: 15% (inrush restraint)

Minimum detectable internal fault (LV): 210 A (20 % of rated)


In [5]:
# LV side is star‑connected with neutral earthed via NER (300 A)
I_ef_max = 300.0    # A — NER design limit
I_ef_min = 0.05 * I_ef_max   # minimum we want to detect (5% of max)

# Typical REF pickup: 10% of LV rated current, but must be > load unbalance current
I_ref_pickup = 0.10 * I_lv_rated
print("=== REF Protection Settings ===")
print(f"  LV rated current: {I_lv_rated:.0f} A")
print(f"  REF pickup (10% of rated): {I_ref_pickup:.0f} A primary")
print(f"  Max earth fault (NER): {I_ef_max} A")
print(f"  Min detectable fault: {I_ref_pickup:.0f} A → covers {(1 - I_ref_pickup/I_ef_max)*100:.0f} % of winding")

# Stability voltage for high-impedance REF
Rct = 3.0      # ohm — CT secondary resistance (typical 1 A CT)
Rl  = 1.5      # ohm — lead resistance (estimate)
If_ext = I_lv_rated * 10    # through-fault current (worst case external)
# Stability voltage: Vs = If_ext_sec * (Rct + 2*Rl)  (for 3‑ph balanced, simplified)
If_ext_sec = If_ext / CT_lv
Vs_min = If_ext_sec * (Rct + 2*Rl)
print(f"\n  Through‑fault current (10× rated): {If_ext:.0f} A primary ({If_ext_sec:.1f} A secondary)")
print(f"  Minimum stability voltage (Vs): {Vs_min:.1f} V")
print("  (Requires metrosil non‑linear resistor to limit voltage during internal fault)")

=== REF Protection Settings ===
  LV rated current: 1050 A
  REF pickup (10% of rated): 105 A primary
  Max earth fault (NER): 300.0 A
  Min detectable fault: 105 A → covers 65 % of winding

  Through‑fault current (10× rated): 10497 A primary (8.7 A secondary)
  Minimum stability voltage (Vs): 52.5 V
  (Requires metrosil non‑linear resistor to limit voltage during internal fault)


### Findings

- The 20 MVA transformer has **no differential protection**. An internal fault would be cleared only by the HV overcurrent, taking >0.8 s and tripping the entire substation.  
- A restrained differential relay (e.g. SEL‑387) with CTs 200/1 HV, 1200/1 LV is recommended.  
- LV REF protection should be added with pickup 105 A and stability voltage ≈13 V (high‑impedance scheme).  

**Deficiency D‑005: No differential or REF protection — RISK HIGH.**  
Work to budget for: relay + CTs + installation.  
Timeframe: Urgent